In [ ]:
import uproot
import pandas as pd

# ─────────────────────────────────────────────────────────────────────
# Load the training ntuple into a pandas DataFrame.
# Each row is one FD π±/K± track passing the script's filters.
# Columns are documented in the script's startup banner (53 total) and
# in the column-map table in Section 4e.
# ─────────────────────────────────────────────────────────────────────
df = uproot.open(
    #"/volatile/clas12/cooperb/SULI/pid_training_test.root:PhysicsEvents"
    "/volatile/clas12/cooperb/SULI/pid_training_v1.root:PhysicsEvents"
).arrays(library="pd")





In [2]:
# Basic sanity check: how many tracks, how many features?
# Expect ~1-2 million rows from one HIPO file, exactly 53 columns.
print("Shape:", df.shape)

# ─────────────────────────────────────────────────────────────────────
# What did the Event Builder reconstruct each track as?
# Column "pid" is the EB-assigned PDG code, filtered by the script to
# {211, -211, 321, -321} (π+, π-, K+, K-).
# ─────────────────────────────────────────────────────────────────────
print("\nEB-assigned PID counts:")
print(df["pid"].value_counts())

Shape: (1096573, 53)

EB-assigned PID counts:
pid
 211    748271
-211    238328
 321     91897
-321     18077
Name: count, dtype: int64


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# What was the MC truth identity of those same tracks?
# Column "mc_matching_pid" is the PDG code of the MC particle that
# geometrically matched the reconstructed track (|Δφ| < 6°, |Δθ| < 2°).
# -9999 means no MC particle matched (rare, <5%).
# Many entries will be non-pion/non-kaon truths — protons, photons,
# decay products of ρ⁰/ω/η, etc. — that EB mis-identified as a pion/kaon.
# ─────────────────────────────────────────────────────────────────────
print("\nMC-truth PID (top 15):")
print(df["mc_matching_pid"].value_counts().head(15))
print(f"\nUnmatched tracks (no MC match): {(df['mc_matching_pid'] == -9999).sum()}")



In [ ]:
# ─────────────────────────────────────────────────────────────────────
# THE KEY DIAGNOSTIC for this project — the contamination matrix.
# Rows = what EB called the track. Columns = what it actually was.
# Off-diagonal entries are the misidentifications ML must learn to fix.
# For Cooper's project the most important row is EB pid=321 (EB-K+):
# how many of those are really K+ (diagonal) vs π+ contaminating (off-diagonal)?
# ─────────────────────────────────────────────────────────────────────
# Show only the most common truth species to keep the table readable
top_truths = df["mc_matching_pid"].value_counts().head(8).index
contam = pd.crosstab(
    df["pid"],
    df["mc_matching_pid"].where(df["mc_matching_pid"].isin(top_truths), other="other"),
    margins=True,
)
print("\nContamination matrix (EB pid × MC-truth pid):")
print(contam)



In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Quick sanity stats on key training features.
# - beta:    should be roughly [0.5, 1.0]. Values > 1 are reco artifacts.
# - chi2pid: should mostly be in [-5, +5]. Sentinel 9999 = EB couldn't compute it.
# - rich_RQ: -9999 means no RICH hit (most tracks — RICH covers one sector only).
# ─────────────────────────────────────────────────────────────────────
print("\nKey feature stats:")
print(df[["beta", "chi2pid", "rich_RQ"]].describe())
print(f"chi2pid sentinels (9999): {(df['chi2pid'] == 9999).sum()}")
print(f"Tracks with RICH hit (rich_RQ != -9999): {(df['rich_RQ'] != -9999).sum()}")